<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/07_lora_finetuning.ipynb)


# colab — LoRA Fine-Tuning of a Small LLM

## _Adapting a 360 M Parameter Model with Low-Rank Adaptation on a Free T4_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Fine-tune a small instruction-following LLM with LoRA on
  a free Colab GPU and compare base vs adapted output.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Data**: 1 000 examples from `yahma/alpaca-cleaned` via
  `datasets.load_dataset`.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Install `transformers`, `datasets`, `peft`,
   `accelerate`, `bitsandbytes`.
2. **Data**: Load a tiny subset of Alpaca and format as instruction text.
3. **Base Model**: Load `SmolLM2-360M-Instruct` in `bfloat16`.
4. **LoRA**: Attach low-rank adapters to the attention layers.
5. **Training**: Fine-tune for 100 steps on GPU.
6. **Comparison**: Generate the same prompt before and after LoRA.

### Environment Setup

We verify the GPU and install the Hugging Face ecosystem.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install transformers datasets peft accelerate
!pip -q install bitsandbytes torchao>=0.16.0
!pip -q install matplotlib

In [ ]:
#@title Imports
import time
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")

### Load and Format Instruction Data

We use 1 000 examples from `yahma/alpaca-cleaned` and format each as
an instruction-following text block.

In [ ]:
#@title Load Alpaca subset
ds = load_dataset(
    "yahma/alpaca-cleaned",
    split="train[:1000]",
)
print(f"Loaded {len(ds)} examples")

def format_example(ex):
    instruction = ex["instruction"]
    inp = ex.get("input", "")
    out = ex["output"]
    if inp:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{inp}\n\n"
            f"### Response:\n{out}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{out}"
        )
    return {"text": text}

ds = ds.map(format_example)
print("\nExample:")
print(ds[0]["text"][:300])

### Load the Base Model and Tokenizer

`SmolLM2-360M-Instruct` is a genuinely small model (~700 MB in
`bfloat16`) that fits comfortably on a T4 and responds well to
short LoRA runs.

In [ ]:
#@title Load model and tokenizer
MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16,
    device_map="auto" if device == "cuda" else None,
).to(device) if device == "cpu" else AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16,
    device_map="auto",
)

print(f"Model on {device}")
if device == "cuda":
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM: {vram:.2f} GB")

### Attach LoRA Adapters

We wrap the base model with rank-8 LoRA adapters targeting the query
and value projection layers. Only ~0.1 % of original parameters are
trainable, yet the adapted model can learn the instruction style.

In [ ]:
#@title Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Base Model Output (Before Fine-Tuning)

Generate an answer from the un-adapted model so we have a baseline
to compare against later.

In [ ]:
#@title Generate baseline response
TEST_PROMPT = (
    "### Instruction:\nExplain what a GPU is and why it matters "
    "for machine learning.\n\n### Response:"
)


def generate(prompt, model, tokenizer, max_new=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(
        model.device
    )
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        out[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True,
    )


base_response = generate(TEST_PROMPT, model, tokenizer)
print("=" * 60)
print("BASE MODEL (before LoRA)")
print("=" * 60)
print(base_response)

### Tokenise and Fine-Tune

We tokenise the instruction dataset and run a short supervised
fine-tuning loop. 100 steps is enough to shift the model’s style.

In [ ]:
#@title Tokenise dataset
def tokenize_fn(ex):
    return tokenizer(
        ex["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )

tok_ds = ds.map(tokenize_fn, batched=True)
tok_ds = tok_ds.remove_columns(["instruction", "input",
                                 "output", "text"])

In [ ]:
#@title Fine-tune with LoRA
args = TrainingArguments(
    output_dir="./lora_output",
    per_device_train_batch_size=4,
    max_steps=100,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="no",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
)

collator = DataCollatorForLanguageModeling(
    tokenizer, mlm=False
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds,
    data_collator=collator,
)

start = time.perf_counter()
trainer.train()
torch.cuda.synchronize() if device == "cuda" else None
elapsed = time.perf_counter() - start
print(f"\nLoRA training time: {elapsed:.1f} s")

### Adapted Model Output (After LoRA)

Generate the same prompt again and compare the style and accuracy
of the answer.

In [ ]:
#@title Generate adapted response
lora_response = generate(TEST_PROMPT, model, tokenizer)
print("=" * 60)
print("ADAPTED MODEL (after LoRA)")
print("=" * 60)
print(lora_response)
print("\n" + "-" * 60)
print(f"Training time: {elapsed:.1f} s")

### Exercises

1. **More steps**: Increase `max_steps` to 500. Does the response
   become more instruction-aligned?
2. **Different model**: Try `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.
   Does it need a different `target_modules` list?
3. **Larger rank**: Increase LoRA rank to 16 or 32. How much longer
   does training take?
4. **Custom data**: Replace the Alpaca subset with your own list of
   50 instruction-response pairs and fine-tune on a niche topic.
5. **Merge adapters**: Use `model.merge_and_unload()` after training
   and measure inference speed vs the adapter-wrapped model.

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
